# Coffee17 W0-HBP V1 — Direct W0 preprocessing + HBP

Eksperimen ini **hanya melatih arm baru W0-HBP**:

`RGB -> W0 (Haar4 + VisuShrink) -> MobileNetV3-Large -> HBP -> Linear17`

Kontrol `R0-HBP` **tidak dilatih ulang**. Notebook mengambil kontrol fold-matched dari output eksperimen HBP-MVFD-SBN yang sudah selesai.

## Kaggle Inputs wajib

1. **Coffee Green Bean with 17 Defects Original**
2. file/output lama yang berisi **`hbp-mvfd-sbn-analysis-package.zip`**

GPU + Internet aktif.

Notebook akan abort bila reference HBP lama tidak cocok dengan validation hashes atau initial HBP fingerprint. Outer test tidak disentuh.


In [ ]:
# Coffee17 W0-HBP V1 — reuse completed HBP-R0 control, train W0-HBP only
SCIENTIFIC_CODE_COMMIT = "f67a1f461dfa40e322068e6a9af0063ba2e99819"

import hashlib, importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-w0-hbp-code"
PROJECT = WORK / "coffee17-w0-hbp-project"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def merge_tree_exact(source, target):
    source, target = Path(source), Path(target)
    for item in sorted(source.rglob("*")):
        if not item.is_file():
            continue
        rel = item.relative_to(source)
        dst = target / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.is_file():
            if sha256_file(item) != sha256_file(dst):
                raise RuntimeError(f"Kaggle input conflict: {rel}")
        else:
            shutil.copy2(item, dst)

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

# Optional resume from output of THIS notebook only.
for prior in sorted(
    p for p in INPUT.rglob("coffee17-w0-hbp-project")
    if p.is_dir()
):
    print("MERGE PRIOR W0-HBP OUTPUT:", prior)
    merge_tree_exact(prior, PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Exact code checkout.
# ------------------------------------------------------------
if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run([
    "git", "-C", REPO, "checkout", "--quiet", "--detach",
    SCIENTIFIC_CODE_COMMIT
])

requirements = REPO / "requirements/preprocessing-study.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")
print("GPU:", torch.cuda.get_device_name(0))

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)
from bilinear_lmmd.engine.shared_multiview import validation_identity_label_sha256

CONFIG = REPO / "configs/w0_hbp/W0_HBP.yaml"
OUT = PROJECT / "experiments/coffee17-w0-hbp-v1"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Build frozen HBP-R0 reference from COMPLETED prior package.
#    No HBP-R0 retraining is allowed.
# ------------------------------------------------------------
print("\n=== 1/5 LOAD COMPLETED HBP-R0 CONTROL ===")

prior_zips = sorted(INPUT.rglob("hbp-mvfd-sbn-analysis-package.zip"))
prior_result_dirs = sorted({
    p.parents[3]
    for p in INPUT.rglob("HBP_MVFD_SBN_ALL4/fold_1/seed42/result.json")
})

if len(prior_zips) > 1:
    raise RuntimeError(
        "Ditemukan lebih dari satu hbp-mvfd-sbn-analysis-package.zip. "
        f"Kandidat: {[str(p) for p in prior_zips]}"
    )
if len(prior_result_dirs) > 1:
    raise RuntimeError(
        "Ditemukan lebih dari satu extracted HBP-MVFD result tree. "
        f"Kandidat: {[str(p) for p in prior_result_dirs]}"
    )
if not prior_zips and not prior_result_dirs:
    raise RuntimeError(
        "Kontrol HBP-R0 lama belum terpasang sebagai Kaggle Input. "
        "Tambahkan output notebook HBP-MVFD-SBN yang sudah selesai. "
        "Notebook menerima salah satu dari: "
        "(1) hbp-mvfd-sbn-analysis-package.zip, atau "
        "(2) extracted tree yang berisi HBP_MVFD_SBN_ALL4/fold_1/seed42/result.json. "
        "Kontrol tidak akan dilatih ulang."
    )

PRIOR_ROOT = WORK / "prior-hbp-mvfd-reference"
shutil.rmtree(PRIOR_ROOT, ignore_errors=True)
PRIOR_ROOT.mkdir(parents=True)

if prior_zips:
    PRIOR_SOURCE = prior_zips[0]
    with zipfile.ZipFile(PRIOR_SOURCE, "r") as zf:
        zf.extractall(PRIOR_ROOT)
    source_package_sha = sha256_file(PRIOR_SOURCE)
    print("PRIOR SOURCE: ZIP", PRIOR_SOURCE)
else:
    PRIOR_SOURCE = prior_result_dirs[0]
    merge_tree_exact(PRIOR_SOURCE, PRIOR_ROOT)
    digest_items = []
    for p in sorted(PRIOR_ROOT.rglob("*")):
        if p.is_file():
            digest_items.append(
                f"{p.relative_to(PRIOR_ROOT).as_posix()}\t{sha256_file(p)}\n"
            )
    source_package_sha = hashlib.sha256(
        "".join(digest_items).encode("utf-8")
    ).hexdigest()
    print("PRIOR SOURCE: EXTRACTED TREE", PRIOR_SOURCE)
reference = {
    "format": "coffee17.w0_hbp.hbp_r0_reference.v1",
    "source": "completed hbp-mvfd-sbn analysis package",
    "source_package_sha256": source_package_sha,
    "control_retrained": False,
    "folds": {},
}

required_metrics = (
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "worst_class_f1",
    "hard_class_f1",
)

initial_hashes = set()
for fold in range(1, 6):
    result_path = (
        PRIOR_ROOT
        / "HBP_MVFD_SBN_ALL4"
        / f"fold_{fold}"
        / "result.json"
    )
    if not result_path.is_file():
        raise RuntimeError(f"Prior HBP result tidak ditemukan: {result_path}")

    result = json.loads(result_path.read_text(encoding="utf-8"))
    if int(result.get("fold", -1)) != fold or int(result.get("seed", -1)) != 42:
        raise RuntimeError(f"Prior HBP fold {fold}: fold/seed mismatch.")
    if result.get("test_images_accessed") is not False:
        raise RuntimeError(f"Prior HBP fold {fold}: test access contract invalid.")

    control = result.get("control_contract", {})
    if control.get("method") != "HBP_R0_CONTROL":
        raise RuntimeError(f"Prior HBP fold {fold}: missing HBP_R0_CONTROL contract.")
    if control.get("deployment_view") != "R0":
        raise RuntimeError(f"Prior HBP fold {fold}: deployment view bukan R0.")
    if control.get("seed") != 42:
        raise RuntimeError(f"Prior HBP fold {fold}: control seed bukan 42.")

    hbp_metrics = result.get("HBP_R0_CONTROL", {})
    if any(k not in hbp_metrics for k in required_metrics):
        raise RuntimeError(f"Prior HBP fold {fold}: metric control tidak lengkap.")

    init_sha = result.get("hbp_initial_model_state_sha256")
    if not init_sha:
        raise RuntimeError(f"Prior HBP fold {fold}: initial fingerprint hilang.")
    initial_hashes.add(init_sha)

    reference["folds"][f"fold_{fold}"] = {
        "count": int(control["validation_count"]),
        "identity_label_sha256": control["validation_identity_label_sha256"],
        "hbp_initial_model_state_sha256": init_sha,
        "HBP_R0_CONTROL": {
            k: float(hbp_metrics[k]) for k in required_metrics
        },
        "source_result_sha256": sha256_file(result_path),
    }

if len(initial_hashes) != 1:
    raise RuntimeError(
        f"Prior HBP initial fingerprint tidak konsisten antar-fold: {initial_hashes}"
    )

REFERENCE = PROJECT / "analysis" / "hbp_r0_reference.json"
REFERENCE.write_text(json.dumps(reference, indent=2) + "\n", encoding="utf-8")

print("PRIOR SOURCE :", PRIOR_SOURCE)
print("PACKAGE SHA  :", source_package_sha)
print("HBP INIT SHA :", next(iter(initial_hashes)))
print("CONTROL      : reused; NOT retrained")

# ------------------------------------------------------------
# 3. Reconstruct exact Coffee17 development folds.
# ------------------------------------------------------------
print("\n=== 2/5 RECONSTRUCT COFFEE17 ===")

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus memuat Coffee17 original valid saja di samping prior ZIP "
        f"(expected 979/17, observed {raw_count}/{len(by_class)})."
    )

ARCHIVE = WORK / "coffee17_w0_hbp_original.zip"
PROV_LOCAL = WORK / "coffee17_w0_hbp_provenance"
CANONICAL = WORK / "coffee17_w0_hbp_original_v1"
FOLDS_LOCAL = WORK / "coffee17_w0_hbp_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE,
    PROV_LOCAL,
    canonical_root=CANONICAL,
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

# ------------------------------------------------------------
# 4. Train ONLY W0-HBP on the five matched development folds.
# ------------------------------------------------------------
print("\n=== 3/5 W0-HBP FIVE FOLDS ===")

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_w0_hbp_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL / "clean_manifest.json",
        FOLDS_LOCAL / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    count, val_sha = validation_identity_label_sha256(DEV)
    expected = reference["folds"][f"fold_{fold}"]
    print("VAL COUNT:", count)
    print("VAL SHA  :", val_sha)
    if count != expected["count"] or val_sha != expected["identity_label_sha256"]:
        raise RuntimeError(f"Fold {fold}: validation authority mismatch.")
    print("COMPLETED HBP-R0 REFERENCE: PASS")

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_w0_hbp",
        "--config", CONFIG,
        "--reference", REFERENCE,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"W0_HBP_fold{fold}.log")

    result = json.loads(
        (
            OUT / "W0_HBP" / f"fold_{fold}" / "seed42" / "result.json"
        ).read_text(encoding="utf-8")
    )
    d = result["DELTA_W0_ON_HBP"]
    print(
        f"FOLD {fold} | "
        f"R0-HBP={result['HBP_R0_CONTROL']['macro_f1']:.4f} | "
        f"W0-HBP={result['W0_HBP']['macro_f1']:.4f} | "
        f"Macro delta={d['macro_f1']:+.4f} | "
        f"Hard delta={d['hard_class_f1']:+.4f} | "
        f"Worst delta={d['worst_class_f1']:+.4f}"
    )

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 5. Summary + compact analysis package.
# ------------------------------------------------------------
print("\n=== 4/5 SUMMARY ===")
SUMMARY = PROJECT / "analysis" / "w0_hbp_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_w0_hbp_summary",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
agg = summary["aggregate"]

print("\n=== KEY RESULT ===")
print("R0-HBP Macro-F1:", f"{agg['HBP_R0_CONTROL']['macro_f1']['mean']:.4%}")
print("W0-HBP Macro-F1:", f"{agg['W0_HBP']['macro_f1']['mean']:.4%}")
print("Delta Macro-F1 :", f"{agg['DELTA_W0_ON_HBP']['macro_f1']['mean']:+.4%}")
print("Delta Hard-F1  :", f"{agg['DELTA_W0_ON_HBP']['hard_class_f1']['mean']:+.4%}")
print("Delta Worst-F1 :", f"{agg['DELTA_W0_ON_HBP']['worst_class_f1']['mean']:+.4%}")
print("Positive Macro folds:", summary["positive_macro_folds"], "/ 5")
print("Positive Hard folds :", summary["positive_hard_folds"], "/ 5")
print("Positive Worst folds:", summary["positive_worst_folds"], "/ 5")
print("SCREENING GATE      :", summary["screening_gate"]["decision"])
print("CONTROL RETRAINED   :", summary["control_retrained"])

print("\n=== 5/5 BUILD ANALYSIS PACKAGE ===")
PKG = WORK / "w0-hbp-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)

shutil.copytree(PROJECT / "analysis", PKG / "analysis")
for fold in range(1, 6):
    src = OUT / "W0_HBP" / f"fold_{fold}" / "seed42"
    dst = PKG / "W0_HBP" / f"fold_{fold}"
    dst.mkdir(parents=True, exist_ok=True)
    for name in ("result.json", "run_contract.json", "history.json", "run_config.yaml"):
        p = src / name
        if p.is_file():
            shutil.copy2(p, dst / name)
    if (src / "validation").is_dir():
        shutil.copytree(src / "validation", dst / "validation")

shutil.copy2(REFERENCE, PKG / "hbp_r0_reference.json")
shutil.copy2(CONFIG, PKG / "W0_HBP.yaml")
protocol = REPO / "docs/protocols/W0_HBP_V1.md"
if protocol.is_file():
    shutil.copy2(protocol, PKG / "W0_HBP_V1.md")

zip_path = Path(
    shutil.make_archive(
        str(WORK / "w0-hbp-analysis-package"),
        "zip",
        root_dir=PKG,
    )
)

print("\nREADY:", zip_path)
print("SIZE :", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("PROJECT:", PROJECT)
print("HBP-R0 control was reused and NOT retrained.")
print("Outer test was NOT accessed.")

for path in (REPO, PRIOR_ROOT, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
